# Embedding Evaluation

Load trained Skip-gram and CBOW models and evaluate their embeddings with:
1. **Most similar words** (cosine similarity)
2. **Word analogies** (e.g. king - man + woman ≈ queen)

In [1]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import numpy as np
from model import SkipGramW2V, CBOWW2V
from dataloader import DataLoader
from datasets import load_dataset

In [2]:
# Rebuild the tokenizer with the same data used for training
data_path = os.path.join(os.getcwd(), 'dataset')
dataset = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', cache_dir=data_path, trust_remote_code=True)
train_dataset = dataset['train']
del dataset

def preprocess_text(text):
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    text = text.replace('.', ' . ').replace(',', ' , ').replace('!', ' ! ').replace('?', ' ? ')
    return text.lower()

train_texts = [preprocess_text(row['text']) for row in train_dataset if row['text'].strip()]
train_text = ' '.join(train_texts)
dl = DataLoader(train_text, vocab_size=10000, context_window=4)
del train_text

tok = dl.tokenizer
print(f'Vocabulary size: {tok.get_vocab_size()}')

Vocabulary size: 10000


In [3]:
# Load trained models
vocab_size = tok.get_vocab_size()
emb_dim = 128

skipgram = SkipGramW2V(vocab_size, emb_dim)
skipgram.load_model('models/skipgram_model.npz')

cbow = CBOWW2V(vocab_size, emb_dim)
cbow.load_model('models/cbow_model.npz')

print(f'Loaded Skip-gram W1: {skipgram.W1.shape}')
print(f'Loaded CBOW      W1: {cbow.W1.shape}')

Loaded Skip-gram W1: (10000, 128)
Loaded CBOW      W1: (10000, 128)


In [4]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-15)

def most_similar(word, model, tokenizer, top_n=10):
    """Find the top_n most similar words to the given word."""
    if word not in tokenizer.token_to_id:
        print(f"'{word}' not in vocabulary")
        return []
    
    word_idx = tokenizer.token_to_id[word]
    word_vec = model.get_embedding(word_idx)
    
    # Compute similarity against all words
    # Normalize all embeddings for fast cosine similarity
    norms = np.linalg.norm(model.W1, axis=1, keepdims=True) + 1e-15
    normed = model.W1 / norms
    word_normed = word_vec / (np.linalg.norm(word_vec) + 1e-15)
    
    sims = normed @ word_normed  # (vocab_size,)
    sims[word_idx] = -1  # exclude the word itself
    
    top_idxs = np.argsort(sims)[-top_n:][::-1]
    results = [(tokenizer.id_to_token[i], sims[i]) for i in top_idxs]
    return results

In [5]:
query_words = ['king', 'love', 'death', 'sword', 'man', 'good', 'lord']

for word in query_words:
    print(f'\n--- {word.upper()} ---')
    print(f'{"Skip-gram":>20}  |  {"CBOW":>20}')
    sg_results = most_similar(word, skipgram, tok)
    cb_results = most_similar(word, cbow, tok)
    for (sw, ss), (cw, cs) in zip(sg_results, cb_results):
        print(f'{sw:>15} {ss:+.3f}  |  {cw:>15} {cs:+.3f}')


--- KING ---
           Skip-gram  |                  CBOW
         hairan +0.930  |           called +0.997
     odaenathus +0.930  |              man +0.997
         gerard +0.927  |            works +0.997
          henry +0.924  |              god +0.997
         gothic +0.917  |            often +0.997
         diable +0.916  |            still +0.996
        william +0.915  |          writing +0.996
        persian +0.913  |          himself +0.996
       medieval +0.912  |             role +0.996
     altarpiece +0.911  |            death +0.996

--- LOVE ---
           Skip-gram  |                  CBOW
         saying +0.973  |              him +0.997
           show +0.964  |             when +0.996
            don +0.963  |             also +0.996
           want +0.963  |              may +0.996
          going +0.962  |            being +0.996
             my +0.962  |             then +0.996
            'll +0.961  |              own +0.996
         really +0.960  |     

In [ ]:
def analogy(a, b, c, model, tokenizer, top_n=5):
    """Solve: a is to b as c is to ?   ->   vec(b) - vec(a) + vec(c)"""
    for word in [a, b, c]:
        if word not in tokenizer.token_to_id:
            print(f"'{word}' not in vocabulary")
            return []
    
    va = model.get_embedding(tokenizer.token_to_id[a])
    vb = model.get_embedding(tokenizer.token_to_id[b])
    vc = model.get_embedding(tokenizer.token_to_id[c])
    
    target = vb - va + vc
    target_normed = target / (np.linalg.norm(target) + 1e-15)
    
    norms = np.linalg.norm(model.W1, axis=1, keepdims=True) + 1e-15
    normed = model.W1 / norms
    sims = normed @ target_normed
    
    # Exclude input words
    for word in [a, b, c]:
        sims[tokenizer.token_to_id[word]] = -1
    
    top_idxs = np.argsort(sims)[-top_n:][::-1]
    return [(tokenizer.id_to_token[i], sims[i]) for i in top_idxs]

In [7]:
analogies = [
    ('man', 'woman', 'king'),       # king - man + woman ≈ queen
    ('good', 'better', 'great'),    # great + (better - good) ≈ ?
    ('father', 'mother', 'son'),    # son + (mother - father) ≈ daughter
    ('lord', 'lady', 'sir'),        # sir + (lady - lord) ≈ ?
]

for a, b, c in analogies:
    print(f'\n"{a}" is to "{b}" as "{c}" is to ...?')
    
    sg_res = analogy(a, b, c, skipgram, tok)
    cb_res = analogy(a, b, c, cbow, tok)
    
    if sg_res and cb_res:
        print(f'  Skip-gram: {[(w, f"{s:.3f}") for w, s in sg_res]}')
        print(f'  CBOW:      {[(w, f"{s:.3f}") for w, s in cb_res]}')


"man" is to "woman" as "king" is to ...?
  Skip-gram: [('witchcraft', '0.931'), ('tradition', '0.925'), ('persian', '0.922'), ('pagan', '0.921'), ('inscriptions', '0.916')]
  CBOW:      [('police', '0.994'), ('co', '0.994'), ('meaning', '0.994'), ('inspired', '0.994'), ('de', '0.994')]

"good" is to "better" as "great" is to ...?
  Skip-gram: [('certainly', '0.970'), ('situation', '0.966'), ('spelling', '0.964'), ('disliked', '0.961'), ('reflected', '0.959')]
  CBOW:      [('majority', '0.991'), ('difficult', '0.991'), ('does', '0.991'), ('james', '0.990'), ('sound', '0.990')]

"father" is to "mother" as "son" is to ...?
  Skip-gram: [('innocent', '0.948'), ('franco', '0.943'), ('congolese', '0.941'), ('soap', '0.940'), ('widow', '0.940')]
  CBOW:      [('itself', '0.995'), ('rather', '0.995'), ('young', '0.995'), ('beyoncé', '0.995'), ('discovered', '0.995')]

"lord" is to "lady" as "sir" is to ...?
  Skip-gram: [('ernest', '0.954'), ('bracknell', '0.944'), ('cecily', '0.938'), ('wif